In [1]:
import pandas as pd
import numpy as np
import yaml
import geopandas as gpd
from shapely.geometry import Point
import re

## Load data

In [31]:
path_to_file = "data/mmc-100.yaml"

with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

df_naselja = pd.read_csv("./processed_data/naselja.csv")

df_regije = df_naselja[['region_id', 'region_name']].drop_duplicates().values

## Find regions for each data

In [23]:
# 1. Create a unique list of all searchable terms (naselje, rodilnik, mestnik)
# We filter out Nones/NaNs to avoid errors
search_terms = set(df_naselja[['naselje', 'rodilnik', 'mestnik']].values.flatten())
search_terms = {str(term) for term in search_terms if term and str(term).lower() != 'nan'}

# 2. Build a regex pattern for efficiency 
# \b ensures we match whole words only
pattern = '|'.join([re.escape(word) for word in search_terms])
regex = re.compile(rf'\b({pattern})\b', flags=re.IGNORECASE)

# 3. Define a helper to find matching regions
def get_intersected_regions(text_list):
    full_text = " ".join(text_list)
    found_words = set(regex.findall(full_text))
    
    if not found_words:
        return None
    
    # Map found words back to their regions in df_naselja
    # We check if any of the three columns match the found words
    mask = (
        df_naselja['naselje'].isin(found_words) | 
        df_naselja['rodilnik'].isin(found_words) | 
        df_naselja['mestnik'].isin(found_words)
    )
    return list(df_naselja.loc[mask, 'region_id'].unique())

# 4. Apply and filter
df['intersected_regions'] = df['paragraphs'].apply(get_intersected_regions)

In [24]:
novice_z_naselji_df = df[df['intersected_regions'].astype(bool)]

In [5]:
novice_z_naselji_df.count()

_id                    31
url                    31
topics                 31
authors                30
date                   31
figures                31
keywords               31
lead                   31
mention                31
paragraphs             31
title                  31
gpt_keywords           22
id                     31
n_comments             27
category                1
intersected_regions    31
dtype: int64

In [25]:
# Spreminjaj i v "novice_z_naselji_df.loc[i]" da vidiš na katere besede proži
x = novice_z_naselji_df.loc[2]["paragraphs"]
for m in x:
	found_words = set(regex.findall(m))
	print(found_words)

set()
{'Javornik'}
set()
set()
{'Ljubljani'}
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()
set()


In [64]:
novice_z_naselji_df_za_db = novice_z_naselji_df[["id", "url", "date", "topics", "paragraphs", "intersected_regions"]]
novice_z_naselji_df_za_db['paragraphs'] = novice_z_naselji_df_za_db['paragraphs'].str.join('\n')

In [65]:
novice_z_naselji_df_za_db.head(3)

,id,url,date,topics,paragraphs,intersected_regions
0,698876,https://www.rtvslo.si/sport/rokomet/liga-prvak...,2024-02-19T21:01:34,sport,Nemške zasedbe po mnenju trenerja Celjanov Ale...,[SI034]
2,728945,https://www.rtvslo.si/gospodarstvo/pasti-crneg...,2024-11-28T20:12:01,gospodarstvo,Če potrošnika po nakupu daje občutek slabe ves...,"[SI043, SI042, SI034, SI041]"
3,729539,https://www.rtvslo.si/sport/nogomet/prva-liga/...,2024-12-04T15:16:43,sport,"Bravo, ki je predtem dosegel tri zaporedne zma...","[SI042, SI037, SI031, SI041, SI032, SI036]"


# Save to db

In [9]:
import sqlite3
print(sqlite3.sqlite_version)

3.45.3


In [67]:
connection = sqlite3.connect('final_data/novice.db') # Naredi db če še ne obstaja
connection.execute("PRAGMA foreign_keys = ON")
cursor = connection.cursor()

In [66]:
connection.close()

In [68]:
# Naredi db
cursor.execute('''
    CREATE TABLE IF NOT EXISTS regije (
        id char(5) PRIMARY KEY,
        name VARCHAR(50)
    )
''')

cursor.executemany("INSERT OR IGNORE INTO regije (id, name) VALUES (?, ?)", df_regije)

cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice (
        id INTEGER PRIMARY KEY,
        url TEXT,
		date DATE,
        topic VARCHAR(30),
        content TEXT
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice_regije (
        novica_id INTEGER,
        regija_id char(5),
        PRIMARY KEY (novica_id, regija_id),
        FOREIGN KEY (novica_id) REFERENCES novice (id) ON DELETE CASCADE,
        FOREIGN KEY (regija_id) REFERENCES regije (id) ON DELETE CASCADE
    )
''')

cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_topic ON novice (topic)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_date ON novice (date)")

connection.commit()
cursor.close()

In [71]:
def save_data_to_db(connection, df):
    cursor = connection.cursor()
    
    # 1. Prepare data for 'novice' table
    # We select only the columns that match the DB schema
    novice_items = df[['id', 'url', 'date','topics', 'paragraphs']].values.tolist()
    
    # 2. Prepare data for 'novice_regije' junction table
    # We iterate through the rows and 'explode' the intersected_regions list
    junction_items = []
    for _, row in df.iterrows():
        n_id = row['id']
        r_ids = row['intersected_regions'] # This is your list of IDs
        
        if isinstance(r_ids, list):
            for r_id in r_ids:
                junction_items.append((n_id, r_id))

    try:
        # Insert into main news table
        cursor.executemany('''
            INSERT OR REPLACE INTO novice (id, url, date, topic, content) 
            VALUES (?, ?, ?, ?, ?)
        ''', novice_items)
        
        # Insert into junction table
        cursor.executemany('''
            INSERT OR IGNORE INTO novice_regije (novica_id, regija_id) 
            VALUES (?, ?)
        ''', junction_items)
        
        connection.commit()
        print(f"Successfully saved {len(df)} articles.")
        
    except Exception as e:
        connection.rollback()
        print(f"Error during save: {e}")
    finally:
        cursor.close()

#! UNCOMMENT IF NEED
save_data_to_db(connection, novice_z_naselji_df_za_db)

Successfully saved 31 articles.
